# 02 — Sweep the 12 bounded ±30° orientation encoders (probe-only, no training)

Amendment A5 excluded the orientation arm. Its **empirical** leg rests on a 3-seed
local check of the ±30° retrain (0.799 / 0.804 / 0.847), while the only full 12-seed
sweep on disk is the **±180°** arm. The 12 bounded encoders were trained 2026-07-25
and never probed.

A8 §b already retro-labels A5's gate paragraph non-load-bearing (A5 stands on its
analytic SO(2)-vs-SO(3) argument), and A7 §d froze the grid at three cells, so this
**cannot reinstate the arm**. It closes the evidentiary gap so the limitations section
reports a 12-seed number instead of a 3-seed one.

No training — probe only, ~3 h. Writes **outside** the confirmatory sweep root, to
`results/probes_excluded/`.

**Setup:** Accelerator `GPU T4 x2`, Internet **On**. Attach the ±30° encoders as an input.

## 1. Verify the GPU(s)

In [ ]:
!nvidia-smi

## 2. Clone the repo
Onto `/kaggle/working` (persists across restarts within a session).

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/kaggle/working/probe-capacity-invariance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Install dependencies
Without disturbing Kaggle's preinstalled, CUDA-matched `torch`/`torchvision`.

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())

## 4. Download shapes3d + build the image cache
`--build-cache` decompresses once into an uncompressed memmap the loaders mmap. Idempotent.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache

## 5. Restore checkpoints from a previous session (resume)
`Add Input -> a prior version's output` (or an uploaded encoder dataset), then run
this. It finds every `orientation_strong_seed*.pt` under `/kaggle/input` and restores it to
`results/encoders/<run_id>/` (`backbone*.pt` -> `backbone.pt`, `last_ckpt*.pt` ->
`last_ckpt.pt`). On a fresh first run there is nothing to restore. Only orientation
files are touched.

In [ ]:
import re, shutil
from pathlib import Path

REPO  = Path("/kaggle/working/probe-capacity-invariance")
ENC   = REPO / "results" / "encoders"
INPUT = Path("/kaggle/input")
_RID  = re.compile(r"orientation_strong_seed\d+")

def _target(name):
    n = name.lower()
    if "ckpt" in n:     return "last_ckpt.pt"
    if "backbone" in n: return "backbone.pt"
    return None

found = {}
for p in sorted(INPUT.rglob("*.pt")) if INPUT.exists() else []:
    m, tgt = _RID.search(p.as_posix()), _target(p.name)
    if m and tgt:
        found.setdefault((m.group(0), tgt), p)     # first match per (run_id, kind)

if not found:
    print("nothing to restore -- fresh start")
for (rid, tgt), src in sorted(found.items()):
    dst = ENC / rid / tgt
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        shutil.copy2(src, dst)
    print(f"{rid:34s} {tgt:14s} <- {src}")

## 6. Integrity check — keep good checkpoints, purge corrupt ones
Each restored `.pt` is opened as a zip. A valid `backbone.pt` means the seed is
done and training skips it. A corrupt `backbone.pt` is deleted with its
`last_ckpt.pt` so the seed retrains; a valid `last_ckpt.pt` with no backbone lets
training resume mid-run.

In [ ]:
import zipfile
from pathlib import Path

ENC = Path("/kaggle/working/probe-capacity-invariance/results/encoders")

def _state(p):
    if not p.exists():
        return "missing"
    try:
        return None if zipfile.ZipFile(p).testzip() is None else "corrupt"
    except Exception as e:
        return f"not-a-zip ({e})"

for d in sorted(ENC.glob("orientation_strong_seed*")):
    bb, ck = d / "backbone.pt", d / "last_ckpt.pt"
    bstat = _state(bb)
    if bstat is None:
        print(f"{d.name:34s} backbone OK -> skip"); continue
    if bstat != "missing":
        bb.unlink(missing_ok=True); ck.unlink(missing_ok=True)
        print(f"{d.name:34s} backbone {bstat} -> purged, will retrain"); continue
    print(f"{d.name:34s} "
          + ("last_ckpt OK -> resume" if _state(ck) is None else "fresh start"))

## 7. Confirm the 12 bounded encoders are present

In [ ]:
import glob
ck = sorted(glob.glob("/kaggle/working/probe-capacity-invariance/results/encoders/orientation_strong_seed*/backbone.pt"))
print(f"{len(ck)} orientation encoders found")
for c in ck[:3]: print(" ", c)
assert len(ck) >= 10, "attach the bounded (+/-30 deg) orientation checkpoints first"

## 8. Probe sweep
`--resume` caches per-seed probe rows under `<out>/_cache`, so a timed-out session
picks up exactly where it stopped.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.run_sweep \
    --config configs/probe/ladder.yaml \
    --dataset shapes3d --condition orientation --strength strong \
    --encoders results/encoders/orientation_strong_seed*/backbone.pt \
    --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 \
    --device cuda --num-workers 2 --resume \
    --out-root results/probes_excluded

## 9. Gate result across 12 seeds

In [ ]:
import json
from pathlib import Path
m = json.loads((Path("/kaggle/working/probe-capacity-invariance")
                / "results/probes_excluded/orientation_strong/meta.json").read_text())
g = m["quality_gate"]
print(f"gate: {g['n_passed']}/{g['n_encoders']} passed  (threshold 0.90)")
for s in g["per_seed"]:
    print(f"  seed {s['seed_index']:2d}  shape={s['shape_recoverability']:.4f}  "
          f"floor={s['floor']:.4f}  {'PASS' if s['passed'] else 'fail'}")

In [ ]:
# --- persist for the next session --------------------------------------------
# /kaggle/working is the notebook's output. Click "Save Version" when this
# finishes, then Add Input -> this output on the next run to resume.
import shutil
from pathlib import Path
src = Path("/kaggle/working/probe-capacity-invariance/results"); dst = Path("/kaggle/working/results")
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)
print(f"persisted {sum(1 for _ in dst.rglob('*') if _.is_file())} files "
      f"-> click 'Save Version' now")